In [3]:
import numpy as np
import itertools
import re


def extract_lrs(s):
    lr_match = re.search(r'_lr([0-9.eE-]+?)(?:[._]|$)', s)
    lr = lr_match.group(1) if lr_match else None

    ti_lr_match = re.search(r'\.ti([0-9.eE-]+?)(?:[._]|$)', s)
    ti_lr = ti_lr_match.group(1) if ti_lr_match else None

    return lr, ti_lr

def extract_learning_lora_rank(s):
    match = re.search(r'c\.l(\d+)\.', s)
    if match:
        return int(match.group(1))
    else:
        return None



def get_chunk(data, chunk_index, total_chunks=4):
    """Get a specific chunk from the data"""
    chunk_size = len(data) // total_chunks
    remainder = len(data) % total_chunks
    
    # Calculate start position
    start = chunk_index * chunk_size + min(chunk_index, remainder)
    
    # Calculate end position
    extra = 1 if chunk_index < remainder else 0
    end = start + chunk_size + extra
    
    return data[start:end]

dataset_name2data_root = {
    'crybabyU3': 'data_root/data/real_data/crybaby/crybaby-unseen-3',
    'crybaby50': 'data_root/data/real_data/crybaby/crybaby-50',
    'moodengU3': 'data_root/data/real_data/moodeng/moodeng-unseen-3',
    'moodeng50': 'data_root/data/real_data/moodeng/moodeng-50',
    'chiquita50': 'data_root/data/real_data/chiquita/chiquita-50',
    'chiquitaU3': 'data_root/data/real_data/chiquita/chiquita-unseen-3',
    'reese50': 'data_root/data/real_data/reese/reese-50',
    'reeseU3': 'data_root/data/real_data/reese/reese-unseen-3',
    'gout50': 'data_root/data/real_data/gout/gout-50',
    'goutU3': 'data_root/data/real_data/gout/gout-unseen-3',
    'jooli50': 'data_root/data/real_data/jooli/jooli-50',
    'jooliU3': 'data_root/data/real_data/jooli/jooli-unseen-3',
    'honer50': 'data_root/data/real_data/honer/honer-50',
    'honerU3': 'data_root/data/real_data/honer/honer-unseen-3',
    'avp20': 'data_root/data/real_data/avp/avp-20',
    'avpS3': 'data_root/data/real_data/avp/avp-seen-3',
}
concept2prompt = {
    'crybaby': 'A photo of a crybaby art toy',
    'moodeng': 'A photo of a cute baby hippo',
}
concept2generalprompt = {
    'crybaby': 'A photo of a toy',
    'moodeng': 'A photo of a hippo',
    
}
concept2initializer = {
    'crybaby': 'toy',
    'moodeng': 'hippo',
    'chiquita': 'person', 
    'reese': 'person', 
    'jooli': 'person', 
    'honer': 'person', 
    'gout': 'person', 
    'avp': 'glasses',
}

concept2Prprompt = {
    'crybaby': 'A photo of a toy',
    'moodeng': 'A photo of a hippo',
    'chiquita': 'A photo of a person',
    'reese': 'A photo of a person',
    'gout': 'A photo of a person',
    'jooli': 'A photo of a person',
    'honer': 'A photo of a person',
    # 'chiquita': 'A photo of a girl',
    'avp': 'A photo of a glasses',
}


# delete

concept2domain_preservation_cache_path = {
    'crybaby': 'data_root/cache/mace/general_concept/cache_crybaby.pt',
    'chiquita':'data_root/cache/mace/cache_cele.pt',
    'reese':'data_root/cache/mace/cache_cele.pt',
    'gout':'data_root/cache/mace/cache_cele.pt',
    'jooli':'data_root/cache/mace/cache_cele.pt',
    'honer':'data_root/cache/mace/cache_cele.pt',
}   

concept2mapping_concept = {
    'crybaby': ['object', 'object'],
    'chiquita': ['person', 'a person'],
    'reese': ['person', 'a person'],
    'gout': ['person', 'a person'],
    'jooli': ['person', 'a person'],
    'honer': ['person', 'a person'],
}



In [ ]:

base_exps = [
    
        "c.l4.kv_honer50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
    "c.l4.kv_honer50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
    "c.l4.kv_honer50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",

    "c.l4.kv_jooli50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
    "c.l4.kv_jooli50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
    "c.l4.kv_jooli50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",


    # "c.l4.kv_reese50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_reese50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_reese50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",

    # "c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_chiquita50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",

    # "c.l4.kv_gout50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_gout50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_gout50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",

    
    # "c.l4.kv_gout50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_gout50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4",
    # "c.l4.kv_gout50-V_pr0.50_lr5e-4.ti5e-3_f0.5_b1g4",
    # "c.l4.kv_gout50-V_pr0.50_lr5e-4.ti1e-3_f0.5_b1g4",
    # "c.l4.kv_gout50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_gout50-V_pr0.50_lr1e-4.ti1e-2_f0.5_b1g4",
    # "c.l4.kv_gout50-V_pr0.50_lr1e-4.ti5e-3_f0.5_b1g4",
    # "c.l4.kv_gout50-V_pr0.50_lr1e-4.ti1e-3_f0.5_b1g4",
    # "c.l4.kv_gout50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_gout50-V_pr0.50_lr5e-5.ti1e-2_f0.5_b1g4",
    # "c.l4.kv_gout50-V_pr0.50_lr5e-5.ti5e-3_f0.5_b1g4",
    # "c.l4.kv_gout50-V_pr0.50_lr5e-5.ti1e-3_f0.5_b1g4",
    # "c.l4.kv_gout50-V_pr0.50_lr1e-5.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_gout50-V_pr0.50_lr1e-5.ti1e-2_f0.5_b1g4",
    # "c.l4.kv_gout50-V_pr0.50_lr1e-5.ti5e-3_f0.5_b1g4",
    # "c.l4.kv_gout50-V_pr0.50_lr1e-5.ti1e-3_f0.5_b1g4",
]

# base_exps = [
#     "c.l4.kv_reese50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_reese50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4",
#     # "c.l4.kv_reese50-V_pr0.50_lr5e-4.ti5e-3_f0.5_b1g4",
#     # "c.l4.kv_reese50-V_pr0.50_lr5e-4.ti1e-3_f0.5_b1g4",
#     "c.l4.kv_reese50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_reese50-V_pr0.50_lr1e-4.ti1e-2_f0.5_b1g4",
#     # "c.l4.kv_reese50-V_pr0.50_lr1e-4.ti5e-3_f0.5_b1g4",
#     # "c.l4.kv_reese50-V_pr0.50_lr1e-4.ti1e-3_f0.5_b1g4",
#     "c.l4.kv_reese50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_reese50-V_pr0.50_lr5e-5.ti1e-2_f0.5_b1g4",
#     # "c.l4.kv_reese50-V_pr0.50_lr5e-5.ti5e-3_f0.5_b1g4",
#     # "c.l4.kv_reese50-V_pr0.50_lr5e-5.ti1e-3_f0.5_b1g4",
#     # "c.l4.kv_reese50-V_pr0.50_lr1e-5.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_reese50-V_pr0.50_lr1e-5.ti1e-2_f0.5_b1g4",
#     # "c.l4.kv_reese50-V_pr0.50_lr1e-5.ti5e-3_f0.5_b1g4",
#     # "c.l4.kv_reese50-V_pr0.50_lr1e-5.ti1e-3_f0.5_b1g4",
# ]

# base_exps = [
    
# "c.l16.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
# "c.l16.kv_chiquita50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
# "c.l16.kv_chiquita50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",

    # "c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
#     "c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4",
#     "c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-3_f0.5_b1g4",
#     "c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-3_f0.5_b1g4",
    # "c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
#     "c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti1e-2_f0.5_b1g4",
#     "c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti5e-3_f0.5_b1g4",
#     "c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti1e-3_f0.5_b1g4",
    # "c.l4.kv_chiquita50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",
#     "c.l4.kv_chiquita50-V_pr0.50_lr5e-5.ti1e-2_f0.5_b1g4",
#     "c.l4.kv_chiquita50-V_pr0.50_lr5e-5.ti5e-3_f0.5_b1g4",
#     "c.l4.kv_chiquita50-V_pr0.50_lr5e-5.ti1e-3_f0.5_b1g4",
#     "c.l4.kv_chiquita50-V_pr0.50_lr1e-5.ti5e-2_f0.5_b1g4",
#     "c.l4.kv_chiquita50-V_pr0.50_lr1e-5.ti1e-2_f0.5_b1g4",
#     "c.l4.kv_chiquita50-V_pr0.50_lr1e-5.ti5e-3_f0.5_b1g4",
#     "c.l4.kv_chiquita50-V_pr0.50_lr1e-5.ti1e-3_f0.5_b1g4",
# ]

# base_exps = [
    # "c.l4.kv_crybaby50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_crybaby50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4",
    # "c.l4.kv_crybaby50-V_pr0.50_lr5e-4.ti5e-3_f0.5_b1g4",
    # "c.l4.kv_crybaby50-V_pr0.50_lr5e-4.ti1e-3_f0.5_b1g4",
    # "c.l4.kv_crybaby50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_crybaby50-V_pr0.50_lr1e-4.ti1e-2_f0.5_b1g4",
    # "c.l4.kv_crybaby50-V_pr0.50_lr1e-4.ti5e-3_f0.5_b1g4",
    # "c.l4.kv_crybaby50-V_pr0.50_lr1e-4.ti1e-3_f0.5_b1g4",
    # "c.l4.kv_crybaby50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_crybaby50-V_pr0.50_lr5e-5.ti1e-2_f0.5_b1g4",
    # "c.l4.kv_crybaby50-V_pr0.50_lr5e-5.ti5e-3_f0.5_b1g4",
    # "c.l4.kv_crybaby50-V_pr0.50_lr5e-5.ti1e-3_f0.5_b1g4",
    # "c.l4.kv_crybaby50-V_pr0.50_lr1e-5.ti5e-2_f0.5_b1g4",
    # "c.l4.kv_crybaby50-V_pr0.50_lr1e-5.ti1e-2_f0.5_b1g4",
    # "c.l4.kv_crybaby50-V_pr0.50_lr1e-5.ti5e-3_f0.5_b1g4",
    # "c.l4.kv_crybaby50-V_pr0.50_lr1e-5.ti1e-3_f0.5_b1g4",
# ]
exp_names = []
domain_preservations = ["8e+3"] # ["8e+2","8e+3","8e-4"] # ["8e+2","8e+3","8e-4"]#  ["8e+3"] # ["8e+3"] # ["8e+4","8e+5"] #  ["8e+2","8e+3","8e-4","8e-5"] # 8.0e+3, 8.0e+4 2e-5 
general_preservations = ["1e-4"]  # ["1e-0","1e-2","1e-4"] # ,"1e-4"] # preservation sclae for the closed-form
learning_rates = ["1e-4"]# ["1e-3", "1e-4", "1e-5"]
num_gen_images = [8] # [50] # [8] 
lora_ranks = [1] # [1]
img_types = ["G"] # ["r","g"]
max_train_steps = [50] # [50,200]
# use_prs = [True] # [True, False]
manual_target_concept = ""

# sur_concept = 'object'
base_exp_steps = [3000] # we want to see it fit first



for base_exp in base_exps:
    
    
    if not manual_target_concept:
        # Try to infer target_concept from base_exp
        possible_concepts = ['moodeng', 'crybaby', 'avp', 'chiquita', 'reese', 'gout', 'jooli', 'honer']
        for concept in possible_concepts:
            if concept in base_exp:
                target_concept = concept
                # print(f"target_concept is not set, inferred and set to '{target_concept}' from base_exp")
                break
    else:
        target_concept = manual_target_concept
        # print(f"target_concept is set to '{target_concept}' manually")
    # else:
    #     print("Warning: target_concept is not set and could not be inferred from base_exp.")




    for base_exp_step in base_exp_steps:
        for lr, num_img, lora_rank, img_type, gen_pr, domain_pr, max_train_step in itertools.product(
            learning_rates, num_gen_images, lora_ranks, img_types, general_preservations,domain_preservations, max_train_steps
        ):
            # print(lr, num_img, lora_rank, img_type, steps)
            
            ul_name  = f'ul{lora_rank}.prg{gen_pr}d{domain_pr}.lr{lr}.n{num_img}.{img_type}'
            exp_name = f"{ul_name}.{target_concept}.{concept2mapping_concept[target_concept][0]}.s{max_train_step}_{base_exp}.s{base_exp_step}"
            
            script = f""" python data_preparation.py configs/custom/erase_default.yaml \\
            exp_name="{exp_name}" \\
            MACE.num_gen_images={num_img} \\
            MACE.lora_weight_dir_path="data_root/logs/{base_exp}/checkpoint-{base_exp_step}" \\
            MACE.token_embedding_dir_path="data_root/logs/{base_exp}/checkpoint-{base_exp_step}" \\
            MACE.input_data_dir="data_root/generated/mace/{base_exp}/checkpoint-{base_exp_step}"
 python training.py configs/custom/erase_default.yaml \\
            exp_name="{exp_name}" \\
            MACE.learning_rate={lr} MACE.max_train_steps={max_train_step} \\
            MACE.rank={lora_rank} \\
            MACE.num_gen_images={num_img} \\
            MACE.domain_preservation_cache_path={concept2domain_preservation_cache_path[target_concept]} MACE.mapping_concept="['{concept2mapping_concept[target_concept][1]}']" \\
            MACE.train_preserve_scale={gen_pr} MACE.preserve_weight={domain_pr} \\
            MACE.input_data_dir="data_root/generated/mace/{base_exp}/checkpoint-{base_exp_step}" \\
            MACE.lora_weight_dir_path="data_root/logs/{base_exp}/checkpoint-{base_exp_step}" \\
            MACE.token_embedding_dir_path="data_root/logs/{base_exp}/checkpoint-{base_exp_step}"
            """

            
            print(script)
            # print(exp_name)
            exp_names += [exp_name]
print(exp_names)

 python data_preparation.py configs/custom/erase_default.yaml \
            exp_name="ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50_c.l4.kv_honer50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000" \
            MACE.num_gen_images=8 \
            MACE.lora_weight_dir_path="data_root/logs/c.l4.kv_honer50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4/checkpoint-3000" \
            MACE.token_embedding_dir_path="data_root/logs/c.l4.kv_honer50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4/checkpoint-3000" \
            MACE.input_data_dir="data_root/generated/mace/c.l4.kv_honer50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4/checkpoint-3000"
 python training.py configs/custom/erase_default.yaml \
            exp_name="ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50_c.l4.kv_honer50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000" \
            MACE.learning_rate=1e-4 MACE.max_train_steps=50 \
            MACE.rank=1 \
            MACE.num_gen_images=8 \
            MACE.domain_preservation_cache_path=data_root/cache/mace/cache_cele.pt MACE.mapping_

In [38]:
# relearning
# decoding unlearning - with same hyperparameter




ul_exp_names = ['ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50_c.l4.kv_honer50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50_c.l4.kv_honer50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4.s3000', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50_c.l4.kv_honer50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4.s3000', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.jooli.person.s50_c.l4.kv_jooli50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.jooli.person.s50_c.l4.kv_jooli50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4.s3000', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.jooli.person.s50_c.l4.kv_jooli50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4.s3000', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.reese.person.s50_c.l4.kv_reese50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.reese.person.s50_c.l4.kv_reese50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4.s3000', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.reese.person.s50_c.l4.kv_reese50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4.s3000', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.chiquita.person.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.chiquita.person.s50_c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4.s3000', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.chiquita.person.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4.s3000']








concept = "gout"
seed = 0
use_pr = True


data_setting = 'full' 

use_manual_params = True
reV = True
is_relearn = True 
manual_params = {
    'data_setting': 'fewshot',
    'lora_rank' :  4
}
final_exp_names = []
for ul_exp_name in ul_exp_names:
    
    base_exp_name = '_'.join(ul_exp_name.split('_')[1:])
    exp_name = base_exp_name
    # print(f"Base Experiment Name: {base_exp_name}")
    lr_lora, lr_ti = extract_lrs(base_exp_name)
    lora_rank = extract_learning_lora_rank(base_exp_name)
    # print(f'lora_rank: {lora_rank}, lr: {lr_lora}, ti_lr: {lr_ti}')
    pretrained_path = f"data_root/logs/{ul_exp_name}/LoRA_fusion_model"

    if not concept:
        if 'moodeng' in exp_name: concept = 'moodeng'
        if 'crybaby' in exp_name: concept = 'crybaby'
        if 'avp' in exp_name: concept = 'avp'
        if 'chiquita' in exp_name: concept = 'chiquita'
        if 'reese' in exp_name: concept = 'reese'
        if 'gout' in exp_name: concept = 'gout'
        if 'jooli' in exp_name: concept = 'jooli'
        if 'honer' in exp_name: concept = 'honer'
    
    
    
    use_ti = 'ti' in exp_name or '-V' in exp_name 
    # use_pr = 'pr' in exp_name


    if data_setting == 'fewshot':
        if concept == 'avp':
            dataset_name = 'avpS3'
        else:
            dataset_name = f'{concept}U3'
    else:
        if concept == 'avp':
            dataset_name = 'avp20'
        else:
            dataset_name = f'{concept}50'
            
    if reV:
        initializer_token = concept2initializer[concept]
    else: 
        initializer_token = ''
        
    if use_ti:
        prompt = 'A photo of a v1'
    else:
        prompt = concept2prompt[concept]

    name_tag = ''
    if is_relearn: name_tag += 'uul'
    name_tag = f'{name_tag} {dataset_name}'
    name_tag += f' l{lora_rank}'
    if use_ti: 
        # name_tag += f' ti.{lr_ti}'
        name_tag += f' ti'

    data_root = dataset_name2data_root[dataset_name]
    
    if use_ti:
        dataset_name_for_exp = dataset_name + "-V"
        # if use_ni:
        #     dataset_name_for_exp += ".ni"
    else: dataset_name_for_exp = dataset_name
    
    
    # renaming to check
    re_exp_name = f'c.l{lora_rank}.kv_{dataset_name_for_exp}'
    if use_pr:
        re_exp_name += f'_pr0.50'
    re_exp_name += '_lr'
    if lora_rank >0: re_exp_name += f"{str(lr_lora)}"
    if use_ti:
        re_exp_name += f'.ti{str(lr_ti)}'
    re_exp_name += '_f0.5_b1g4'
    

    
    # print(re_exp_name)
    # assert re_exp_name in base_exp_name, f"Expected {re_exp_name} in {base_exp_name}"

    # if manual_lora is not None and manual_data != lora_rank:
    
    if use_manual_params:
        lora_rank = manual_params['lora_rank']
        eff_data_setting = manual_params['data_setting']
        
        if eff_data_setting == 'fewshot':
            if concept == 'avp':
                dataset_name = 'avpS3'
            else:
                dataset_name = f'{concept}U3'
        else:
            if concept == 'avp':
                dataset_name = 'avp20'
            else:
                dataset_name = f'{concept}50'
        data_root = dataset_name2data_root[dataset_name]
    
    if reV:
        relearn_exp_name = f"rl{lora_rank}.reV.{dataset_name}"
    else:
        relearn_exp_name = f"rl{lora_rank}.{dataset_name}"
        
        
    if not use_pr:
        relearn_exp_name += f".pr0.00"
        
    if seed != 0:
        relearn_exp_name += f".r{seed}"

    final_exp_name = f"{relearn_exp_name}_{ul_exp_name}"
    
    
    script = f"""
    accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path={pretrained_path}  \\
    --instance_data_dir={data_root} \\
    --output_dir="data_root/logs/{final_exp_name}" \\
    --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
    --train_batch_size=1 --gradient_accumulation_steps=4 \\
    --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
    --max_train_steps=1000  --validation_steps=250  --checkpointing_steps=50 --seed {seed} \\
    --run_note '{name_tag}' \\"""
        
        
    if use_pr:
        script += f"""
    --with_prior_preservation --prior_loss_weight=0.5 --num_class_images 50 \\
    --class_prompt="{concept2Prprompt[concept]}" --class_data_dir="data_root/generated/model/original_pretrained/{concept2Prprompt[concept]}/7.50" \\"""
        
    # Conditional learning rate + TI options
    if use_ti:
        
        if lora_rank <= 0:
            script += f"""
    --learning_rate_ti {lr_ti} \\
    --placeholder_token="v1" --initializer_token='{initializer_token}'"""
        else:
            script += f"""
    --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
    --placeholder_token="v1" --initializer_token='{initializer_token}'"""
    else:
        script += f"""
    --learning_rate {lr_lora}"""

    print(script)
    final_exp_names += [final_exp_name]
print(final_exp_names)

        


    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path=data_root/logs/ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50_c.l4.kv_honer50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000/LoRA_fusion_model  \
    --instance_data_dir=data_root/data/real_data/gout/gout-unseen-3 \
    --output_dir="data_root/logs/rl4.reV.goutU3_ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50_c.l4.kv_honer50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000" \
    --validation_prompt="A photo of a v1" --instance_prompt="A photo of a v1" \
    --train_batch_size=1 --gradient_accumulation_steps=4 \
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
    --max_train_steps=1000  --validation_steps=250  --checkpointing_steps=50 --seed 0 \
    --run_note 'uul gout50 l4 ti' \
    --with_prior_preservation --prior_loss_weight=0.5 --num_class_images 50 \
    --class_prompt="A photo of a person" --class_data_dir="data_root/generated/model/original_pretrained/A photo of a person/7.50" \
  

In [39]:
# exp_names = ['rl4.chiquita50_ul1.prg1e-4d8e-5.lr1e-4.n8.G.chiquita.person.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000']
#    ['uul1.lr1e-4.n8.G.chiquita.obj.s8_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000', 
    # 'uul1.lr1e-4.n8.G.chiquita.obj.s0_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000']
# ul_exp_names = , 'ul1.lr1e-4.n8.G.chiquita.obj.s0_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-3_f0.5_b1g4.s3000']
# ul_exp_names = [, ]
# uul1.prg8e+7d1e-2.lr1e-4.n8.G.chiquita.obj.s0_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4.s3000
exp_names = ['rl4.reV.goutU3_ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50_c.l4.kv_honer50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000', 'rl4.reV.goutU3_ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50_c.l4.kv_honer50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4.s3000', 'rl4.reV.goutU3_ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50_c.l4.kv_honer50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4.s3000', 'rl4.reV.goutU3_ul1.prg1e-4d8e+3.lr1e-4.n8.G.jooli.person.s50_c.l4.kv_jooli50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000', 'rl4.reV.goutU3_ul1.prg1e-4d8e+3.lr1e-4.n8.G.jooli.person.s50_c.l4.kv_jooli50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4.s3000', 'rl4.reV.goutU3_ul1.prg1e-4d8e+3.lr1e-4.n8.G.jooli.person.s50_c.l4.kv_jooli50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4.s3000', 'rl4.reV.goutU3_ul1.prg1e-4d8e+3.lr1e-4.n8.G.reese.person.s50_c.l4.kv_reese50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000', 'rl4.reV.goutU3_ul1.prg1e-4d8e+3.lr1e-4.n8.G.reese.person.s50_c.l4.kv_reese50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4.s3000', 'rl4.reV.goutU3_ul1.prg1e-4d8e+3.lr1e-4.n8.G.reese.person.s50_c.l4.kv_reese50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4.s3000', 'rl4.reV.goutU3_ul1.prg1e-4d8e+3.lr1e-4.n8.G.chiquita.person.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000', 'rl4.reV.goutU3_ul1.prg1e-4d8e+3.lr1e-4.n8.G.chiquita.person.s50_c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4.s3000', 'rl4.reV.goutU3_ul1.prg1e-4d8e+3.lr1e-4.n8.G.chiquita.person.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4.s3000']



cfg_scales = [  4.5,6.0, 7.5]


for exp_name in exp_names:
    # manual_prompt = 'A photo of a toy'# 'A photo of a toy'
    # manual_prompt = 'A photo of a hippo'
    
    
    is_relearn = 'uul' in exp_name or 'rl' in exp_name
    if is_relearn:
        base_exp_name = '_'.join(exp_name.split('_')[2:])
        relearn_exp_name = exp_name
        unlearn_exp_name = '_'.join(exp_name.split('_')[1:])
        exp_name = base_exp_name

    
    manual_prompt = ''
    use_general_concept = False
    # cfg_scales = np.arange(2.0,4.5, 0.5).tolist()
    # cfg_scales = np.arange(3.0,3.5, 0.5).tolist()

    # steps = [50,100,150,200]
    # for step in steps:
    for step in range(0, 1000+1, 100):
    # for step in [2000]:

        for cfg in cfg_scales:
            is_original_pretrained = exp_name == 'CompVis/stable-diffusion-v1-4'
            is_unlearn = 'ul' in exp_name and not is_relearn
            if 'moodeng' in exp_name: concept = 'moodeng'
            if 'crybaby' in exp_name: concept = 'crybaby'
            if 'avp' in exp_name: concept = 'avp'
            if 'chiquita' in exp_name: concept = 'chiquita'
            
            pretrained_path = 'CompVis/stable-diffusion-v1-4'
            if is_relearn:
                pretrained_path = f"data_root/logs/{unlearn_exp_name}/LoRA_fusion_model"
                # erase_name = concept
                # if 'VPr' in exp_name: erase_name += 'VPr'
                # pretrained_path = f"data_root/logs/erase_l1.{erase_name}.object_lr2.5e-4/LoRA_fusion_model"
            if is_unlearn: 
                pretrained_path = f"data_root/logs/{exp_name}/LoRA_fusion_model"

            use_ti = 'ti' in exp_name or '-V' in exp_name 
            
            if is_relearn and not 'reV' in relearn_exp_name:
                # relearn is not re-initializing the token (by default)
                initializer_token = ''
            elif use_ti:
                initializer_token = concept2initializer[concept]

            if manual_prompt:
                prompt = manual_prompt
            elif use_general_concept:
                prompt = concept2generalprompt[concept]
            
            elif use_ti:
                prompt = 'A photo of a v1'
            else:
                prompt = concept2prompt[concept]
                
                
            ## hacky .. should change this later
            if is_relearn:
                exp_name = relearn_exp_name
            if is_unlearn or 'erase' in exp_name or exp_name == 'original_pretrained': 
                load_lora_weight_path = ''
                gen_image_path = f"data_root/generated/model/{exp_name}"
            else:
                load_lora_weight_path =f"data_root/logs/{exp_name}/checkpoint-{step}"
                gen_image_path = 'auto'
                
            # if 'l0' in exp_name :
            #     load_lora_weight_path = ''
            
            script = f"""
            accelerate launch train_dreambooth_lora.py \\
                --pretrained_model_name_or_path='{pretrained_path}'  \\
                --instance_data_dir="data_root/data/real_data/dummy" \\
                --load_lora_weight_path="{load_lora_weight_path}" \\
                --gen_image_path="{gen_image_path}" \\
                --output_dir="data_root/logs/gen" \\
                --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
                --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
                --run_note 'gen img' --wait_weight \\
                --num_validation_images 50 \\"""
                
                    
            if use_ti and not is_unlearn:
                script += f"""
                --load_token_embedding_path="data_root/logs/{exp_name}/checkpoint-{step}" \\
                --placeholder_token="v1" --initializer_token='{initializer_token}' \\"""

            script += f"""
                --cfg_scale {cfg:.2f}"""
        
            print(script) 
        
        


            accelerate launch train_dreambooth_lora.py \
                --pretrained_model_name_or_path='data_root/logs/ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50_c.l4.kv_honer50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000/LoRA_fusion_model'  \
                --instance_data_dir="data_root/data/real_data/dummy" \
                --load_lora_weight_path="data_root/logs/rl4.reV.goutU3_ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50_c.l4.kv_honer50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000/checkpoint-0" \
                --gen_image_path="auto" \
                --output_dir="data_root/logs/gen" \
                --validation_prompt="A photo of a v1" --instance_prompt="A photo of a v1" \
                --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
                --run_note 'gen img' --wait_weight \
                --num_validation_images 50 \
                --load_token_embedding_path="data_root/logs/rl4.reV.goutU3_ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50_

In [67]:
exp_names = [
    "c.l4.kv_chiquitaU3-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
]



    # decoding exp_name to gneration script 

    # exp_name ="uul.l1.moodengVPr.object_c.l4.kv_moodeng50-V_lr2.5e-4.ti1e-2_f0.5_b1g4"
for exp_name in exp_names:
    # manual_prompt = 'A photo of a toy'# 'A photo of a toy'
    # manual_prompt = 'A photo of a hippo'
    manual_prompt = ''
    use_general_concept = False
    # cfg_scales = np.arange(2.0,4.5, 0.5).tolist()
    cfg_scales = np.arange(3.0,3.5, 0.5).tolist()

    cfg_scales = [4.5,6.0,7.5]
    # steps = [50,100,150,200]
    # for step in steps:
    # for step in range(3100, 4000+1, 100):
    for step in [2000]:
    # for step in range(0, 3000+1, 100):

    # for step in range(300, 1001, 100):
        for cfg in cfg_scales:
            is_original_pretrained = exp_name == 'CompVis/stable-diffusion-v1-4'
            is_relearn = ('uul' in exp_name) or ('erase' in exp_name)
            
            if 'moodeng' in exp_name: concept = 'moodeng'
            if 'crybaby' in exp_name: concept = 'crybaby'
            if 'avp' in exp_name: concept = 'avp'
            if 'chiquita' in exp_name: concept = 'chiquita'
            
            
            pretrained_path = 'CompVis/stable-diffusion-v1-4'
            if is_relearn:
                erase_name = concept
                if 'VPr' in exp_name: erase_name += 'VPr'
                pretrained_path = f"data_root/logs/erase_l1.{erase_name}.object_lr2.5e-4/LoRA_fusion_model"

            use_ti = 'ti' in exp_name or '-V' in exp_name
            
            if 'V.ni' in exp_name:
                initializer_token = ''
            elif use_ti:
                initializer_token = concept2initializer[concept]



            if manual_prompt:
                prompt = manual_prompt
            elif use_general_concept:
                prompt = concept2generalprompt[concept]
            
            elif use_ti:
                prompt = 'A photo of a v1'
            else:
                prompt = concept2prompt[concept]
            
            if 'erase' in exp_name or exp_name == 'original_pretrained': 
                load_lora_weight_path = ''
                gen_image_path = f"data_root/generated/model/{exp_name}"
            else:
                load_lora_weight_path =f"data_root/logs/{exp_name}/checkpoint-{step}"
                gen_image_path = 'auto'
                
            if 'l0' in exp_name :
                load_lora_weight_path = ''
            
            script = f"""
            accelerate launch train_dreambooth_lora.py \\
                --pretrained_model_name_or_path='{pretrained_path}'  \\
                --instance_data_dir="data_root/data/real_data/dummy" \\
                --load_lora_weight_path="{load_lora_weight_path}" \\
                --gen_image_path="{gen_image_path}" \\
                --output_dir="data_root/logs/gen" \\
                --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
                --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
                --run_note 'gen img' --wait_weight \\
                --num_validation_images 50 \\"""
                
                    
            if use_ti:
                script += f"""
                --load_token_embedding_path="data_root/logs/{exp_name}/checkpoint-{step}" \\
                --placeholder_token="v1" --initializer_token='{initializer_token}' \\"""

            script += f"""
                --cfg_scale {cfg:.2f}"""
        
            print(script) 
        
        


            accelerate launch train_dreambooth_lora.py \
                --pretrained_model_name_or_path='CompVis/stable-diffusion-v1-4'  \
                --instance_data_dir="data_root/data/real_data/dummy" \
                --load_lora_weight_path="data_root/logs/c.l4.kv_chiquitaU3-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4/checkpoint-2000" \
                --gen_image_path="auto" \
                --output_dir="data_root/logs/gen" \
                --validation_prompt="A photo of a v1" --instance_prompt="A photo of a v1" \
                --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
                --run_note 'gen img' --wait_weight \
                --num_validation_images 50 \
                --load_token_embedding_path="data_root/logs/c.l4.kv_chiquitaU3-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4/checkpoint-2000" \
                --placeholder_token="v1" --initializer_token='girl' \
                --cfg_scale 4.50

            accelerate launch train_dreambooth_lora

In [69]:

# decoding unlearning - with same hyperparameter
ul_exp_names = [
    "ul4.lr1e-4.n50.G.chiquita.obj.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000",
    
    
    # "ul4.lr1e-4.n50.G.chiquita.obj.s200_c.l4.kv_chiquita50-V_pr0.50_lr1e-4.ti1e-2_f0.5_b1g4.s3000"
]
data_setting = 'full' 
use_ni = True
for ul_exp_name in ul_exp_names:
    
    base_exp_name = '_'.join(ul_exp_name.split('_')[1:])
    exp_name = base_exp_name
    # print(f"Base Experiment Name: {base_exp_name}")
    
    lr_lora, lr_ti = extract_lrs(base_exp_name)
    lora_rank = extract_learning_lora_rank(base_exp_name)
    # print(f'lora_rank: {lora_rank}, lr: {lr_lora}, ti_lr: {lr_ti}')
    pretrained_path = f"data_root/logs/{ul_exp_name}/LoRA_fusion_model"

    if 'moodeng' in exp_name: concept = 'moodeng'
    if 'crybaby' in exp_name: concept = 'crybaby'
    if 'avp' in exp_name: concept = 'avp'
    if 'chiquita' in exp_name: concept = 'chiquita'
    use_ti = 'ti' in exp_name or '-V' in exp_name 
    use_pr = 'pr' in exp_name


    if data_setting == 'fewshot':
        if concept == 'avp':
            dataset_name = 'avpS3'
        else:
            dataset_name = f'{concept}U3'
    else:
        if concept == 'avp':
            dataset_name = 'avp20'
        else:
            dataset_name = f'{concept}50'

            
    if use_ni:
        initializer_token = ''
    elif use_ti:
        initializer_token = concept2initializer[concept]
        
    if use_ti:
        prompt = 'A photo of a v1'
    else:
        prompt = concept2prompt[concept]

    name_tag = ''
    if is_relearn: name_tag += 'uul'
    name_tag = f'{name_tag} {dataset_name}'
    name_tag += f' l{lora_rank}'
    if use_ti: 
        # name_tag += f' ti.{lr_ti}'
        name_tag += f' ti'

    data_root = dataset_name2data_root[dataset_name]
    
    if use_ti:
        dataset_name_for_exp = dataset_name + "-V"
        # if use_ni:
        #     dataset_name_for_exp += ".ni"
    else: dataset_name_for_exp = dataset_name
    
    
    # renaming to check
    re_exp_name = f'c.l{lora_rank}.kv_{dataset_name_for_exp}'
    if use_pr:
        re_exp_name += f'_pr0.50'
    re_exp_name += '_lr'
    if lora_rank >0: re_exp_name += f"{str(lr_lora)}"
    if use_ti:
        re_exp_name += f'.ti{str(lr_ti)}'
    re_exp_name += '_f0.5_b1g4'
    
    # print(re_exp_name)
    assert re_exp_name in base_exp_name, f"Expected {re_exp_name} in {base_exp_name}"

    script = f"""
    accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path={pretrained_path}  \\
    --instance_data_dir={data_root} \\
    --output_dir="data_root/logs/u{ul_exp_name}" \\
    --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
    --train_batch_size=1 --gradient_accumulation_steps=4 \\
    --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
    --max_train_steps=3000  --validation_steps=250  --checkpointing_steps=50 \\
    --run_note '{name_tag}' \\"""
        
        
    if use_pr:
        script += f"""
    --with_prior_preservation --prior_loss_weight=0.5 --num_class_images 50 \\
    --class_prompt="{concept2Prprompt[concept]}" --class_data_dir="data_root/generated/model/original_pretrained/{concept2Prprompt[concept]}/7.50" \\"""
        
    # Conditional learning rate + TI options
    if use_ti:
        
        if lora_rank <= 0:
            script += f"""
    --learning_rate_ti {lr_ti} \\
    --placeholder_token="v1" --initializer_token='{initializer_token}'"""
        else:
            script += f"""
    --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
    --placeholder_token="v1" --initializer_token='{initializer_token}'"""
    else:
        script += f"""
    --learning_rate {lr_lora}"""

    print(script)




        


    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path=data_root/logs/ul4.lr1e-4.n50.G.chiquita.obj.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000/LoRA_fusion_model  \
    --instance_data_dir=data_root/data/real_data/chiquita/chiquita-50 \
    --output_dir="data_root/logs/uul4.lr1e-4.n50.G.chiquita.obj.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000" \
    --validation_prompt="A photo of a v1" --instance_prompt="A photo of a v1" \
    --train_batch_size=1 --gradient_accumulation_steps=4 \
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
    --max_train_steps=3000  --validation_steps=250  --checkpointing_steps=50 \
    --run_note ' chiquita50 l4 ti' \
    --with_prior_preservation --prior_loss_weight=0.5 --num_class_images 50 \
    --class_prompt="A photo of a girl" --class_data_dir="data_root/generated/model/original_pretrained/A photo of a girl/7.50" \
    --learning_rate_lora 5e-4 --learning_rat

In [5]:


concept = 'jooli' # moodeng
data_setting = 'fewshot' # full
is_relearn = False # True

lora_rank = 4 # 1
use_pr = False
use_ti = True # True 
use_ni = False


seeds = [0]  # List of seeds

lr_lora_grid = ["5e-4", "1e-4", "5e-5"]
lr_ti_grid   = ["5e-2"]   # only used if use_ti

# Create all combinations of lr_lora, lr_ti, and seed
combos = list(itertools.product(lr_lora_grid,
                                lr_ti_grid if use_ti else [None],
                                seeds))
chunk_id = 0
total_chunks = 1
combos = get_chunk(combos, chunk_id, total_chunks=total_chunks)

final_exp_names = []
for lr_lora, lr_ti, seed in combos:

    if data_setting == 'fewshot':
        if concept == 'avp':
            dataset_name = 'avpS3'
        else:
            dataset_name = f'{concept}U3'
    else:
        if concept == 'avp':
            dataset_name = 'avp20'
        else:
            dataset_name = f'{concept}50'


    # however, if use_ti is True, the prompt will be changed to 'A photo of a v1' for all concepts
    data_root = dataset_name2data_root[dataset_name]
    if use_ti:
        prompt = 'A photo of a v1' 
    else:
        prompt = concept2prompt[concept]
        
    pretrained_path = 'CompVis/stable-diffusion-v1-4' 
    if  is_relearn:
        pretrained_path = f"data_root/logs/erase_l1.{concept}VPr.object_lr2.5e-4/LoRA_fusion_model"

            
    if use_ti:
        dataset_name_for_exp = dataset_name + "-V"
        if use_ni:
            dataset_name_for_exp += ".ni"
    else: dataset_name_for_exp = dataset_name

    exp_name = f'c.l{lora_rank}.kv_{dataset_name_for_exp}'
    if use_pr:
        exp_name += f'_pr0.50'
    exp_name += '_lr'
    if lora_rank >0: exp_name += f"{str(lr_lora)}"
    if use_ti:
        exp_name += f'.ti{str(lr_ti)}'
    exp_name += '_f0.5_b1g4'
    if is_relearn:
        unlearn_setting = pretrained_path.split("/")[-2].split("_")[1]
        exp_name = f'uul.{unlearn_setting}_{exp_name}'
        
    if use_ni: initializer_token = ''
    else: 
        initializer_token = concept2initializer[concept]



    name_tag = ''
    if is_relearn: name_tag += 'uul'
    name_tag = f'{name_tag} {dataset_name}'
    name_tag += f' l{lora_rank}'
    if use_ti: 
        # name_tag += f' ti.{lr_ti}'
        name_tag += f' ti'


    max_train_steps = 3000
    if data_setting == 'fewshot':
        max_train_steps = 1000
        
    if seed != 0:
        exp_name += f'.r{seed}'
        name_tag += f' r{seed}'
    
    script = f"""
    accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path={pretrained_path}  \\
    --instance_data_dir={data_root} \\
    --output_dir="data_root/logs/{exp_name}" \\
    --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
    --train_batch_size=1 --gradient_accumulation_steps=4 \\
    --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
    --max_train_steps={max_train_steps}  --validation_steps=250  --checkpointing_steps=50 --seed {seed} \\
    --run_note '{name_tag}' \\"""
        
        
    if use_pr:
        script += f"""
    --with_prior_preservation --prior_loss_weight=0.5 --num_class_images 50 \\
    --class_prompt="{concept2Prprompt[concept]}" --class_data_dir="data_root/generated/model/original_pretrained/{concept2Prprompt[concept]}/7.50" \\"""
        

    # Conditional learning rate + TI options
    if use_ti:
        
        if lora_rank <= 0:
            script += f"""
    --learning_rate_ti {lr_ti} \\
    --placeholder_token="v1" --initializer_token='{initializer_token}'"""
        else:
            script += f"""
    --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
    --placeholder_token="v1" --initializer_token='{initializer_token}'"""
    else:
        script += f"""
    --learning_rate {lr_lora}"""

    print(script)
    # print(exp_name)

    final_exp_names += [exp_name]
print(final_exp_names)


    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path=CompVis/stable-diffusion-v1-4  \
    --instance_data_dir=data_root/data/real_data/jooli/jooli-unseen-3 \
    --output_dir="data_root/logs/c.l4.kv_jooliU3-V_lr5e-4.ti5e-2_f0.5_b1g4" \
    --validation_prompt="A photo of a v1" --instance_prompt="A photo of a v1" \
    --train_batch_size=1 --gradient_accumulation_steps=4 \
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
    --max_train_steps=1000  --validation_steps=250  --checkpointing_steps=50 --seed 0 \
    --run_note ' jooliU3 l4 ti' \
    --learning_rate_lora 5e-4 --learning_rate_ti 5e-2 \
    --placeholder_token="v1" --initializer_token='person'

    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path=CompVis/stable-diffusion-v1-4  \
    --instance_data_dir=data_root/data/real_data/jooli/jooli-unseen-3 \
    --output_dir="data_root/logs/c.l4.kv_jooliU3-V_lr1e-4.ti5e-2_f0.5_b1g4" \
    --va

In [ ]:
# # this one
base_exps = ['c.l4.kv_jooliU3-V_lr5e-4.ti5e-2_f0.5_b1g4', 'c.l4.kv_jooliU3-V_lr1e-4.ti5e-2_f0.5_b1g4', 'c.l4.kv_jooliU3-V_lr5e-5.ti5e-2_f0.5_b1g4']

# base_exps = [
#     "c.l4.kv_goutU3-V_lr5e-4.ti5e-2_f0.5_b1g4.r2",
#     "c.l4.kv_goutU3-V_lr1e-4.ti5e-2_f0.5_b1g4.r2",
#     "c.l4.kv_goutU3-V_lr5e-5.ti5e-2_f0.5_b1g4.r2",
#     # "c.l4.kv_goutU3-V_lr5e-4.ti5e-2_f0.5_b1g4.r1",
#     # "c.l4.kv_goutU3-V_lr1e-4.ti5e-2_f0.5_b1g4.r1",
#     # "c.l4.kv_goutU3-V_lr5e-5.ti5e-2_f0.5_b1g4.r1",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_chiquitaU3-V_lr5e-4.ti5e-2_f0.5_b1g4.r2",
#     # "c.l4.kv_chiquitaU3-V_lr1e-4.ti5e-2_f0.5_b1g4.r2",
#     # "c.l4.kv_chiquitaU3-V_lr5e-5.ti5e-2_f0.5_b1g4.r2",
#     # "c.l4.kv_reeseU3-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.r2",
#     # "c.l4.kv_reeseU3-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4.r2",
#     # "c.l4.kv_reeseU3-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4.r2",
#     # "c.l4.kv_reeseU3-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.r1",
#     # "c.l4.kv_reeseU3-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4.r1",
#     # "c.l4.kv_reeseU3-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4.r1",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.r3",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4.r3",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4.r3",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.r2",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4.r2",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4.r2",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.r1",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4.r1",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4.r1",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_jooliU3-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_jooliU3-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_jooliU3-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_honerU3-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_honerU3-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_honerU3-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_honer50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_honer50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_honer50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_jooli50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_jooli50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_jooli50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_goutU3-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_goutU3-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_goutU3-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_gout50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_gout50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_gout50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_reeseU3-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_reeseU3-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_reeseU3-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_reese50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_reese50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_reese50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",
#     # "c.l16.kv_chiquitaU3-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
#     # "c.l16.kv_chiquitaU3-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
#     # "c.l16.kv_chiquitaU3-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",
#     # "c.l16.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
#     # "c.l16.kv_chiquita50-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
#     # "c.l16.kv_chiquita50-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr5e-4.ti5e-3_f0.5_b1g4",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr5e-4.ti1e-3_f0.5_b1g4",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr1e-4.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr1e-4.ti1e-2_f0.5_b1g4",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr1e-4.ti5e-3_f0.5_b1g4",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr1e-4.ti1e-3_f0.5_b1g4",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr5e-5.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr5e-5.ti1e-2_f0.5_b1g4",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr5e-5.ti5e-3_f0.5_b1g4",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr5e-5.ti1e-3_f0.5_b1g4",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr1e-5.ti5e-2_f0.5_b1g4",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr1e-5.ti1e-2_f0.5_b1g4",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr1e-5.ti5e-3_f0.5_b1g4",
#     # "c.l4.kv_chiquitaU3-V_pr0.50_lr1e-5.ti1e-3_f0.5_b1g4",
# ]

# base_exps = [base_exps[i] for i in range(0, len(base_exps), 4)] # take every second element


chunk_id = 0
total_chunks=1
exp_names = get_chunk(base_exps,chunk_id,total_chunks=total_chunks)


    # decoding exp_name to gneration script 

    # exp_name ="uul.l1.moodengVPr.object_c.l4.kv_moodeng50-V_lr2.5e-4.ti1e-2_f0.5_b1g4"
for exp_name in exp_names:
    # manual_prompt = 'A photo of a toy'# 'A photo of a toy'
    # manual_prompt = 'A photo of a hippo'
    manual_prompt = ''
    use_general_concept = False
    # cfg_scales = np.arange(2.0,4.5, 0.5).tolist()
    cfg_scales = np.arange(3.0,3.5, 0.5).tolist()

    cfg_scales = [3.0,4.5,6.0,7.5]
    # steps = [50,100,150,200]
    # for step in steps:
    for step in range(0, 1001, 100):
    # for step in [2000]:
    # for step in range(0, 3000+1, 100):

    # for step in range(300, 1001, 100):
        for cfg in cfg_scales:
            is_original_pretrained = exp_name == 'original_pretrained'
            is_relearn = ('uul' in exp_name) or ('erase' in exp_name)
            is_unlearn = 'ul' in exp_name and not 'uul' in exp_name
            if 'moodeng' in exp_name: concept = 'moodeng'
            if 'crybaby' in exp_name: concept = 'crybaby'
            if 'avp' in exp_name: concept = 'avp'
            if 'chiquita' in exp_name: concept = 'chiquita'
            if 'reese' in exp_name: concept = 'reese'
            if 'gout' in exp_name: concept = 'gout'
            if 'jooli' in exp_name: concept = 'jooli'
            if 'honer' in exp_name: concept = 'honer'
            
            
            pretrained_path = 'CompVis/stable-diffusion-v1-4'
            if is_relearn:
                erase_name = concept
                if 'VPr' in exp_name: erase_name += 'VPr'
                pretrained_path = f"data_root/logs/erase_l1.{erase_name}.object_lr2.5e-4/LoRA_fusion_model"
            if is_unlearn: 
                pretrained_path = f"data_root/logs/{exp_name}/LoRA_fusion_model"

            use_ti = 'ti' in exp_name or '-V' in exp_name 
            # print(f"use_ti: {use_ti}")
            # print(exp_name)
            
            if 'V.ni' in exp_name:
                initializer_token = ''
            elif use_ti:
                initializer_token = concept2initializer[concept]



            if manual_prompt:
                prompt = manual_prompt
            elif use_general_concept:
                prompt = concept2generalprompt[concept]
            
            elif use_ti:
                prompt = 'A photo of a v1'
            else:
                prompt = concept2prompt[concept]
            
            if is_unlearn or 'erase' in exp_name or exp_name == 'original_pretrained': 
                load_lora_weight_path = ''
                gen_image_path = f"data_root/generated/model/{exp_name}"
            else:
                load_lora_weight_path =f"data_root/logs/{exp_name}/checkpoint-{step}"
                gen_image_path = 'auto'
                
            if 'l0' in exp_name :
                load_lora_weight_path = ''
            
            script = f"""
            accelerate launch train_dreambooth_lora.py \\
                --pretrained_model_name_or_path='{pretrained_path}'  \\
                --instance_data_dir="data_root/data/real_data/dummy" \\
                --load_lora_weight_path="{load_lora_weight_path}" \\
                --gen_image_path="{gen_image_path}" \\
                --output_dir="data_root/logs/gen" \\
                --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
                --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
                --run_note 'gen img' --wait_weight \\
                --num_validation_images 50 \\"""
                
                    
            if use_ti and not is_unlearn:
                script += f"""
                --load_token_embedding_path="data_root/logs/{exp_name}/checkpoint-{step}" \\
                --placeholder_token="v1" --initializer_token='{initializer_token}' \\"""

            script += f"""
                --cfg_scale {cfg:.2f}"""
        
            print(script) 
        
        


            accelerate launch train_dreambooth_lora.py \
                --pretrained_model_name_or_path='CompVis/stable-diffusion-v1-4'  \
                --instance_data_dir="data_root/data/real_data/dummy" \
                --load_lora_weight_path="data_root/logs/c.l4.kv_goutU3-V_lr5e-4.ti5e-2_f0.5_b1g4.r2/checkpoint-0" \
                --gen_image_path="auto" \
                --output_dir="data_root/logs/gen" \
                --validation_prompt="A photo of a v1" --instance_prompt="A photo of a v1" \
                --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
                --run_note 'gen img' --wait_weight \
                --num_validation_images 50 \
                --load_token_embedding_path="data_root/logs/c.l4.kv_goutU3-V_lr5e-4.ti5e-2_f0.5_b1g4.r2/checkpoint-0" \
                --placeholder_token="v1" --initializer_token='person' \
                --cfg_scale 3.00

            accelerate launch train_dreambooth_lora.py \
              